<a href="https://colab.research.google.com/github/Samraat-22/first-project/blob/main/cat_dog_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.preprocessing import image
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.system('pip install kaggle -q')
os.system('kaggle datasets download -d shaunthesheep/microsoft-catsvsdogs-dataset')
os.system('unzip -q microsoft-catsvsdogs-dataset.zip -d dataset')
print("Dataset ready!")

removed = 0
for folder in ['Cat', 'Dog']:
    path = f'dataset/PetImages/{folder}'
    for fname in os.listdir(path):
        fpath = os.path.join(path, fname)
        try:
            img = Image.open(fpath)
            img.verify()
        except:
            os.remove(fpath)
            removed += 1
print(f" {removed} corrupted images removed")

base_model = tf.keras.applications.VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
for layer in base_model.layers[:-4]:
    layer.trainable = False
for layer in base_model.layers[-4:]:
    layer.trainable = True

inputs  = tf.keras.layers.Input(shape=(224, 224, 3))
x       = base_model(inputs, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dense(256, activation='relu')(x)
x       = tf.keras.layers.BatchNormalization()(x)
x       = tf.keras.layers.Dropout(0.5)(x)
x       = tf.keras.layers.Dense(128, activation='relu')(x)
x       = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model   = tf.keras.models.Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print(" Model ready!")

train_datagen = ImageDataGenerator(
    rescale=1./255, validation_split=0.2,
    horizontal_flip=True, zoom_range=0.3,
    rotation_range=20, width_shift_range=0.2,
    height_shift_range=0.2, shear_range=0.2,
    brightness_range=[0.8, 1.2]
)
val_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = train_datagen.flow_from_directory(
    'dataset/PetImages', target_size=(224, 224),
    batch_size=32, class_mode='binary', subset='training', seed=42)
val_data = val_datagen.flow_from_directory(
    'dataset/PetImages', target_size=(224, 224),
    batch_size=32, class_mode='binary', subset='validation', seed=42)
print(" Data ready!")

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1)
]

history = model.fit(train_data, validation_data=val_data, epochs=10, callbacks=callbacks)
print(" Training complete!")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy')
ax1.legend()
ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss')
ax2.legend()
plt.tight_layout()
plt.show()

def predict_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    x   = image.img_to_array(img) / 255.0
    x   = np.expand_dims(x, axis=0)
    pred = model.predict(x)[0][0]
    label = " Dog" if pred > 0.5 else " Cat"
    confidence = pred * 100 if pred > 0.5 else (1 - pred) * 100
    print(f"{label}  —  {confidence:.1f}% confident")

from google.colab import files
uploaded = files.upload()
for img_name in uploaded.keys():
    predict_image(img_name)

model.save('cat_dog_v3.h5')
from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy('cat_dog_v3.h5', '/content/drive/MyDrive/cat_dog_v3.h5')
print(" Model saved to Drive!")

print(" Generating confusion matrix...")
val_data.reset()
preds = (model.predict(val_data) > 0.5).astype(int).flatten()
y_true = val_data.classes
class_names = list(val_data.class_indices.keys())

cm = confusion_matrix(y_true, preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=0.5, linecolor='white',
            annot_kws={"size": 16, "weight": "bold"})
plt.xlabel('Predicted Label', fontsize=13)
plt.ylabel('Actual Label', fontsize=13)
plt.title('Confusion Matrix — Cat vs Dog', fontsize=15)
plt.tight_layout()
plt.show()

print(classification_report(y_true, preds, target_names=class_names))
print(" Done!")